In [3]:
import ray
import time
import pytz
from datetime import datetime
from IPython.display import clear_output

if ray.is_initialized():
    ray.shutdown()

ray.init(address="auto", namespace="GFW_DOWNLOAD")

monitor = ray.get_actor("GFW_Worker")

while True:
    try:
        status, errors = ray.get(monitor.check_status.remote(), timeout=10)
        
        clear_output(wait=True)

        print("Errors:")
        for error in errors:
            print(error)
            
        tz = pytz.timezone('Europe/Zurich')
        current_time = datetime.now(tz).strftime('%H:%M:%S')
        
        print(f"Update [{current_time}]:")
        for key, value in status.items():
            print(f"  {key:12}: {value}")

    except ray.exceptions.GetTimeoutError:
        print(f"[{time.strftime('%H:%M:%S')}] Actor is busy (waiting for API)...")
    except Exception as e:
        print(f"Monitoring stopped: {e}")
        break
    
    time.sleep(1)

2026-03-26 10:27:11,982	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-03-26 10:27:11,984	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver


ValueError: Failed to look up actor with name 'GFW_Worker'. This could because 1. You are trying to look up a named actor you didn't create. 2. The named actor died. 3. You did not use a namespace matching the namespace of the actor.